In [ ]:
%pip uninstall -y onnxruntime-gpu -q
%pip install onnxruntime -q

In [ ]:
"""import yaml 

tracker_settings = {
    "tracker_type": "deepocsort",
    "track_high_thresh": 0.5,
    "track_low_thresh": 0.1,
    "new_track_thresh": 0.6,
    "track_buffer": 90,
    "match_thresh": 0.8,
    "fuse_score": True,
    "proximity_thresh": 0.5,  # Nesnelerin birbirine yakınlık eşiği
    "appearance_thresh": 0.92, # Görünüm benzerlik eşiği (0 ile 1 arası)
    "gmc_method": "sparseOptFlow",
    "with_reid": True,
    "model": "auto"
}
config_path = "/content/drive/MyDrive/deepocsort.yaml"
with open(config_path, 'w') as f:
    yaml.dump(tracker_settings, f)
print(f"dosya başarıyla oluşturuldu: {config_path}")"""


In [ ]:
import yaml 

tracker_settings = {
    "tracker_type": "tracktrack",
    "track_high_thresh": 0.5,       # Biraz düşürdük, nesneleri daha rahat yakalasın
    "track_low_thresh": 0.1,
    "new_track_thresh": 0.6,        # Yeni ID atamadan önce daha emin olsun
    "track_buffer": 150,             # Biri arkada kaybolduğunda 90 kare (3 sn) onu unutmasın
    "match_thresh": 0.8,
    "gmc_method": "sparseOptFlow",
    "with_reid": True,              # Görsel hafıza AÇIK
    "model": "auto"  
                                  # Hata almamak için YOLO'nun kendi altyapısını kullan
}

config_path = "deepocsort.yaml"
with open(config_path, 'w') as f:
    yaml.dump(tracker_settings, f)
print(f"TrackTrack ayar dosyası başarıyla oluşturuldu: {config_path}")

In [ ]:
%pip install ultralytics

In [ ]:
import cv2 as cv
from ultralytics import YOLO
import datetime

model = YOLO('day-04/yolov8m.pt')

video_path = 'day-04/people_in_the_cafe.mp4'
cap = cv.VideoCapture(video_path)

frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv.CAP_PROP_FPS))

fourcc = cv.VideoWriter_fourcc(*'mp4v')

time_stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

file_name = f"cafe_detection_deepocsort_1_{time_stamp}.mp4"

out =cv.VideoWriter(file_name,fourcc,fps,(frame_width,frame_height))
if not out.isOpened():
    raise RuntimeError("VideoWriter açılamadı! Codec veya path sorunu var.")

try:
    while cap.isOpened():
        ret,frame = cap.read()
        if not ret:
            print("video işleme tamamlandı!")
            break    

        #modeli çalıştır => classes=[0] sadece insanları filtreler. ve conf = 0.5 parametresi %50'den düşük doğruluk oranlarını eler 
        #döngü öncesinde modeli ve video ayarlarını hazırlamalıyız. 
        
        results = model.track(
            frame,
            tracker = "deepocsort.yaml",  #'deepocsort.yaml' dosyasını belirtiyoruz
            classes= [0],            #sadece insanlar
            conf = 0.3, 
            persist=True,           #ID'leri hafızada tutması için
            iou=0.6,
            imgsz = 960)

        #üzerine çizim yapacağımız boş kareyi oluşturuyorum
        annotative_frame = frame.copy()

        #eğer ekranda insan bulunduysa ve onlara ID atandıysa
        if results[0].boxes.id is not None:

            #kutuların koordinatlarını ve atanan ID numaralarını listeye çeviriyo
            boxes = results[0].boxes.xyxy.int().cpu().tolist()
            ids = results[0].boxes.id.int().cpu().tolist()

            #her bir insan için çizim
            for box,track_id in zip(boxes,ids):
                x1, y1, x2, y2 = box

                cv.rectangle(annotative_frame, (x1,y1), (x2,y2),(0,255,0), 2)
                etiket = f"Target {track_id}"
                cv.putText(annotative_frame, etiket,(x1, y1 - 10), cv.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        
        out.write(annotative_frame)        #işlenmiş olan kareyi oluşan video içine yazıyorum.
finally:
    cap.release()
    out.release()
    cv.destroyAllWindows()